In [ ]:
import numpy as np

from typing import Union, List
from pymatching import Matching
import operators as o

In [ ]:
def detect_changes(syndrome: list) -> list:
    syndrome = np.array(syndrome)

    syndrome_change = np.abs(np.diff(syndrome, axis=0, prepend=np.zeros((1, syndrome.shape[1]))))

    return syndrome_change


def build_matching_graph(rounds, num_stabilizers, px, pm):
    matching = Matching()

    if px > 0:
        weightx = -np.log(px/(1-px))
    if pm > 0:
        weightm = -np.log(pm/(1-pm))


    for i in range(rounds):
        
        # Adding physical edges
        if px > 0:
            matching.add_boundary_edge(num_stabilizers*i, weight=weightx)
            matching.add_boundary_edge(num_stabilizers*(i + 1) - 1, weight=weightx)
            for s in range(num_stabilizers - 1):
                matching.add_edge((num_stabilizers*i)+s, (num_stabilizers*i)+s+1, weight=weightx)
            

        # Adding measurement edges (currently no time boundry TODO)
        if pm > 0:
            if i < rounds - 1:
                for s in range(num_stabilizers):
                    matching.add_edge((num_stabilizers*i)+s, (num_stabilizers*(i+1))+s, weight=weightm)

    return matching


def match_errors(matching_graph: Matching, detection_events) -> list:

    matched = matching_graph.decode_to_matched_dets_array(detection_events.flatten())
    
    return matched


def extract_predicted_errors(matched_errors, num_stabilizers, num_data_qubits):
    errors = np.zeros(num_data_qubits)
    for match in matched_errors:
        # TODO. This works specifically for 3 bit repitition code with bitflip noise
        if -1 not in match:
            s1, t1 = match[0] %  num_stabilizers, match[0] // num_stabilizers
            s2, t2 = match[1] % num_stabilizers, match[1] // num_stabilizers

            if s1 - s2 != 0:
                m1, m2 = min(s1, s2), max(s1, s2)
                for i in range(m1, m2):
                    errors[i + 1] += 1
        else:
            for i in match:
                if i != -1:
                    s = i % num_data_qubits

                    if s == 0:
                        errors[0] += 1
                    else:
                        errors[-1] += 1

    return errors % 2


rels = {
    ('X', 'Z'): '-Y',
    ('Z', 'X'): 'Y',
    ('X', 'Y'): 'Z',
    ('Y', 'X'): '-Z',
    ('Y', 'Z'): 'X',
    ('Z', 'Y'): '-X',
}

# TODO: Check assumptions
def get_syndrome_bit(state, stabilizer):
    bit = 0
    for comp in zip(state, stabilizer):
        if 'I' not in comp:
            if comp[0] != comp[1]:
                bit += 1

    return bit % 2

In [ ]:
rounds = 20
shots = 2000
px = 0.04
pm = 0.01

intial_state = o.Operator(3)

S1 = o.Z(3, [0, 1])
S2 = o.Z(3, [1, 2])

# stabs = [0, 0]
matching_graph = build_matching_graph(rounds, 2, px, pm)
# er = {}


result = 0
for shot in range(shots):
    syndrome = []
    true_phys_error = [0, 0, 0]
    state = intial_state.copy()
    for rnd in range(rounds):

        
        for i in range(3):
            if np.random.random() < px:
                state = state * o.X(3, i)
                true_phys_error[i] += 1

        sb1 = get_syndrome_bit(state, S1)
        sb2 = get_syndrome_bit(state, S2)

        syndrome.append([sb1, sb2])

    
    detections = detect_changes(np.array(syndrome))
    # print(detections)
    matches = match_errors(matching_graph, detections)
    predicted_errors = extract_predicted_errors(matches)

    true_total_error = [i % 2 for i in true_phys_error]
    
    diff = sum(np.abs(true_total_error - predicted_errors))
    
    if diff != 0:
        result += 1

print(result/shots)
        

In [ ]:
rounds = 20
shots = 20000
px = 0.04
pm = 0.01

intial_state = o.Operator(9)

stabs = {
    'S1': o.Z(9, [0, 1]),
    'S2': o.Z(9, [1, 2]),
    'S3': o.Z(9, [3, 4]),
    'S4': o.Z(9, [4, 5]),
    'S5': o.Z(9, [6, 7]),
    'S6': o.Z(9, [7, 8]),
    'S7': o.X(9, [1, 2, 3, 4, 5, 6]),
    'S8': o.X(9, [3, 4, 5, 6, 7, 8]),
    }

num_stabilizers = 8

matching_graph = build_matching_graph(rounds, num_stabilizers, px, pm)
# print(matching_graph)
# er = {}


result = 0
for shot in range(shots):
    syndrome = []
    true_phys_error = np.zeros(9)
    state = intial_state.copy()
    for rnd in range(rounds):
        # TODO: can do this more efficiently        
        for i in range(9):
            if np.random.random() < px:
                # print('hit', i)
                state = state * o.X(9, i)
                true_phys_error[i] += 1

        t_measurements = []
        for stab in stabs.keys():
            t_measurements.append(get_syndrome_bit(state, stabs[stab]))
        
        syndrome.append(t_measurements)

    
    detections = detect_changes(np.array(syndrome))
    # print(detections)
    matches = match_errors(matching_graph, detections)
    # print(matches)
    predicted_errors = extract_predicted_errors(matches, num_stabilizers, 9)
    # print(predicted_errors)

    true_total_error = [i % 2 for i in true_phys_error]
    
    diff = sum(np.abs(true_total_error - predicted_errors))
    
    if diff != 0:
        result += 1

print(result/shots)

In [ ]:
#Surface code
3*3

data_qubits = o.Operator(2, 3, 2, 3)

# Maybe dont even need an initial state? Maybe don't need to measure stabilizers
# Can just directly add noise and generate what that would look like in the detectors/syndrome
# Generate blank syndrome
# [[0, 0, 0,]] 
# [[0, 0, 0,]]
# [[0, 0, 0,]]

# [[1, 1, 0,]] z11
# [[0, 0, 1,]] z10
# [[0, 0, 1,]]